In [ ]:
# In[1]:

import pandas as pd
import pytz

# Load CSVs (they are in current working directory)
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")

# Row counts
metric_row_count = metric_df.shape[0]
trace_row_count = trace_df.shape[0]

# Distinct cmdb_id lists (as small series)
metric_cmdb_ids = pd.Series(sorted(metric_df['cmdb_id'].dropna().unique()), name='metric_cmdb_ids')
trace_cmdb_ids = pd.Series(sorted(trace_df['cmdb_id'].dropna().unique()), name='trace_cmdb_ids')

# Helper to convert epoch-second series to Asia/Shanghai tz-aware datetimes
tz = pytz.timezone('Asia/Shanghai')
def to_shanghai_dt(s):
    # s is integer/float epoch seconds
    return pd.to_datetime(s, unit='s', utc=True).dt.tz_convert(tz)

# Metric: summary by kpi_name
metric_group = metric_df.groupby('kpi_name', dropna=False)['timestamp']
metric_kpi_summary = metric_group.agg(count_of_rows='count', min_ts='min', max_ts='max').reset_index()
metric_kpi_summary['min_timestamp'] = to_shanghai_dt(metric_kpi_summary['min_ts'])
metric_kpi_summary['max_timestamp'] = to_shanghai_dt(metric_kpi_summary['max_ts'])
metric_kpi_summary = metric_kpi_summary[['kpi_name', 'count_of_rows', 'min_timestamp', 'max_timestamp']]
metric_kpi_summary = metric_kpi_summary.sort_values('count_of_rows', ascending=False).head(20)

# Trace: summary by trace_name
trace_group = trace_df.groupby('trace_name', dropna=False)['timestamp']
trace_trace_summary = trace_group.agg(count_of_rows='count', min_ts='min', max_ts='max').reset_index()
trace_trace_summary['min_timestamp'] = to_shanghai_dt(trace_trace_summary['min_ts'])
trace_trace_summary['max_timestamp'] = to_shanghai_dt(trace_trace_summary['max_ts'])
trace_trace_summary = trace_trace_summary[['trace_name', 'count_of_rows', 'min_timestamp', 'max_timestamp']]
trace_trace_summary = trace_trace_summary.sort_values('count_of_rows', ascending=False).head(20)

# Samples (up to 10 rows) with timestamp converted to Asia/Shanghai
metric_sample = metric_df[['timestamp', 'cmdb_id', 'kpi_name', 'value']].copy().head(10)
metric_sample['timestamp'] = to_shanghai_dt(metric_sample['timestamp'])

trace_sample = trace_df[['timestamp', 'cmdb_id', 'trace_name', 'value']].copy().head(10)
trace_sample['timestamp'] = to_shanghai_dt(trace_sample['timestamp'])

# Final compact outputs (variables displayed by separating with commas)
metric_row_count, trace_row_count, metric_cmdb_ids, trace_cmdb_ids, metric_kpi_summary, trace_trace_summary, metric_sample, trace_sample

```
Out[1]:
```
```python
# Compose a concise plain-English summary using the previously computed variables
metric_ids_sample = metric_cmdb_ids.head(10).tolist()
trace_ids_sample = trace_cmdb_ids.head(10).tolist()

# Top 5 metric KPIs from the prepared summary
metric_top5_lines = []
for _, row in metric_kpi_summary.head(5).iterrows():
    metric_top5_lines.append(f"- {row['kpi_name']}: {row['count_of_rows']} rows, {row['min_timestamp']} -> {row['max_timestamp']}")

# Top 5 trace names from the prepared summary
trace_top5_lines = []
for _, row in trace_trace_summary.head(5).iterrows():
    trace_top5_lines.append(f"- {row['trace_name']}: {row['count_of_rows']} rows, {row['min_timestamp']} -> {row['max_timestamp']}")

summary = (
    "Overview of loaded telemetry files:\n\n"
    f"1) metric.csv: {metric_row_count} rows total.\n"
    f"   Distinct cmdb_id count: {metric_cmdb_ids.shape[0]}. Example ids (up to 10): {metric_ids_sample}\n\n"
    f"2) trace.csv: {trace_row_count} rows total.\n"
    f"   Distinct cmdb_id count: {trace_cmdb_ids.shape[0]}. Example ids (up to 10): {trace_ids_sample}\n\n"
    "3) Top metric KPI summary (top 5 shown):\n"
    + "\n".join(metric_top5_lines) + "\n\n"
    "4) Top trace summary (top 5 shown):\n"
    + "\n".join(trace_top5_lines) + "\n\n"
    "5) Sample rows from metric.csv (up to 10) and 6) sample rows from trace.csv (up to 10) were prepared separately.\n"
    "   (Samples show timestamps in Asia/Shanghai timezone and were included in the earlier output.)"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(44398, 6444, 0        db_001
1        db_002
2        db_003
3        db_004
4        db_005
        ...    
51    redis_008
52    redis_009
53    redis_010
54    redis_011
55    redis_012
Name: metric_cmdb_ids, Length: 56, dtype: object, 0    docker_001
1    docker_002
2    docker_003
3    docker_004
4    docker_005
5    docker_006
6    docker_007
7    docker_008
8        os_021
9        os_022
Name: trace_cmdb_ids, dtype: object,                     kpi_name  count_of_rows             min_timestamp             max_timestamp
19              Disk_io_util            662 2020-05-27 00:00:00+08:00 2020-05-27 00:30:00+08:00
72                Send_total            661 2020-05-27 00:00:00+08:00 2020-05-27 00:30:00+08:00
67                Recv_total            660 2020-05-27 00:00:00+08:00 2020-05-27 00:29:00+08:00
57                   Page_po            659 2020-05-27 00:00:00+08:00 2020-05-27 00:29:00+08:00
3                 Agent_ping            659 2020-05-27 00:00:00+08:00 2020-05-27 00:29:00+08:00
20               Disk_rd_ios            659 2020-05-27 00:00:00+08:00 2020-05-27 00:30:00+08:00
33  Incoming_network_traffic            659 2020-05-27 00:00:00+08:00 2020-05-27 00:30:00+08:00
8               CPU_idle_pct            659 2020-05-27 00:00:00+08:00 2020-05-27 00:29:00+08:00
47           Memory_used_pct            658 2020-05-27 00:00:00+08:00 2020-05-27 00:29:00+08:00
32                 ICMP_ping            658 2020-05-27 00:00:00+08:00 2020-05-27 00:29:00+08:00
46               Memory_used            658 2020-05-27 00:00:00+08:00 2020-05-27 00:29:00+08:00
9            CPU_iowait_time            658 2020-05-27 00:00:00+08:00 2020-05-27 00:29:00+08:00
42          Memory_available            658 2020-05-27 00:00:00+08:00 2020-05-27 00:29:00+08:00
21               Disk_rd_kbs            658 2020-05-27 00:00:00+08:00 2020-05-27 00:29:00+08:00
63      Processor_load_5_min            657 2020-05-27 00:00:00+08:00 2020-05-27 00:30:00+08:00
62      Processor_load_1_min            657 2020-05-27 00:00:00+08:00 2020-05-27 00:29:00+08:00
53  Outgoing_network_traffic            657 2020-05-27 00:00:00+08:00 2020-05-27 00:30:00+08:00
23               Disk_wr_ios            656 2020-05-27 00:00:00+08:00 2020-05-27 00:29:00+08:00
56                   Page_pi            656 2020-05-27 00:00:00+08:00 2020-05-27 00:29:00+08:00
10           CPU_system_time            656 2020-05-27 00:00:00+08:00 2020-05-27 00:29:00+08:00,                                            trace_name  count_of_rows             min_timestamp             max_timestamp
47                          trace.self.jdbc.row_count            240 2020-05-27 00:00:00+08:00 2020-05-27 00:29:00+08:00
46                         trace.self.jdbc.error_rate            240 2020-05-27 00:00:00+08:00 2020-05-27 00:29:00+08:00
45                       trace.self.jdbc.duration_p95            240 2020-05-27 00:00:00+08:00 2020-05-27 00:29:00+08:00
44                      trace.self.jdbc.duration_mean            240 2020-05-27 00:00:00+08:00 2020-05-27 00:29:00+08:00
51                         trace.self.local.row_count            240 2020-05-27 00:00:00+08:00 2020-05-27 00:29:00+08:00
50                        trace.self.local.error_rate            240 2020-05-27 00:00:00+08:00 2020-05-27 00:29:00+08:00
49                      trace.self.local.duration_p95            240 2020-05-27 00:00:00+08:00 2020-05-27 00:29:00+08:00
48                     trace.self.local.duration_mean            240 2020-05-27 00:00:00+08:00 2020-05-27 00:29:00+08:00
39                           trace.self.csf.row_count            180 2020-05-27 00:00:00+08:00 2020-05-27 00:29:00+08:00
38                          trace.self.csf.error_rate            180 2020-05-27 00:00:00+08:00 2020-05-27 00:29:00+08:00
37                        trace.self.csf.duration_p95            180 2020-05-27 00:00:00+08:00 2020-05-27 00:29:00+08:00
36                       trace.self.csf.duration_mean            180 2020-05-27 00:00:00+08:00 2020-05-27 00:29:00+08:00
43                     trace.self.flyremote.row_count            120 2020-05-27 00:00:00+08:00 2020-05-27 00:29:00+08:00
42                    trace.self.flyremote.error_rate            120 2020-05-27 00:00:00+08:00 2020-05-27 00:29:00+08:00
41                  trace.self.flyremote.duration_p95            120 2020-05-27 00:00:00+08:00 2020-05-27 00:29:00+08:00
40                 trace.self.flyremote.duration_mean            120 2020-05-27 00:00:00+08:00 2020-05-27 00:29:00+08:00
0   trace.from_docker_001.remoteprocess.duration_mean             60 2020-05-27 00:00:00+08:00 2020-05-27 00:29:00+08:00
1    trace.from_docker_001.remoteprocess.duration_p95             60 2020-05-27 00:00:00+08:00 2020-05-27 00:29:00+08:00
2      trace.from_docker_001.remoteprocess.error_rate             60 2020-05-27 00:00:00+08:00 2020-05-27 00:29:00+08:00
3       trace.from_docker_001.remoteprocess.row_count             60 2020-05-27 00:00:00+08:00 2020-05-27 00:29:00+08:00,                   timestamp cmdb_id             kpi_name      value
0 2020-05-27 00:00:00+08:00  db_001                  ACS   0.020000
1 2020-05-27 00:00:00+08:00  db_001                 AIOS   0.000000
2 2020-05-27 00:00:00+08:00  db_001                  AWS   0.020000
3 2020-05-27 00:00:00+08:00  db_001         CPU_Used_Pct   1.049194
4 2020-05-27 00:00:00+08:00  db_001         CPU_free_pct  98.950806
5 2020-05-27 00:00:00+08:00  db_001         Call_Per_Sec  13.100000
6 2020-05-27 00:00:00+08:00  db_001  DFParaWrite_Per_Sec   0.000000
7 2020-05-27 00:00:00+08:00  db_001               DbTime   5.600000
8 2020-05-27 00:00:00+08:00  db_001         Exec_Per_Sec  11.400000
9 2020-05-27 00:00:00+08:00  db_001                 Hang   1.000000,                   timestamp     cmdb_id                                     trace_name       value
0 2020-05-27 00:00:00+08:00  docker_001  trace.from_os_022.remoteprocess.duration_mean    0.243944
1 2020-05-27 00:00:00+08:00  docker_001   trace.from_os_022.remoteprocess.duration_p95    0.467200
2 2020-05-27 00:00:00+08:00  docker_001     trace.from_os_022.remoteprocess.error_rate    0.000000
3 2020-05-27 00:00:00+08:00  docker_001      trace.from_os_022.remoteprocess.row_count   90.000000
4 2020-05-27 00:00:00+08:00  docker_001                   trace.self.csf.duration_mean    0.035236
5 2020-05-27 00:00:00+08:00  docker_001                    trace.self.csf.duration_p95    0.090100
6 2020-05-27 00:00:00+08:00  docker_001                      trace.self.csf.error_rate    0.000000
7 2020-05-27 00:00:00+08:00  docker_001                       trace.self.csf.row_count  360.000000
8 2020-05-27 00:00:00+08:00  docker_001             trace.self.flyremote.duration_mean    0.007511
9 2020-05-27 00:00:00+08:00  docker_001              trace.self.flyremote.duration_p95    0.007000)```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz
import datetime

# Reuse metric_df if present; otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# Ensure numeric values
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Timezone setup (UTC+8)
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 and P5 per (cmdb_id, kpi_name) using entire series (before filtering)
global_q = metric_df.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
    global_P5=lambda x: x.quantile(0.05),
    global_P95=lambda x: x.quantile(0.95)
).reset_index()

pairs_count = global_q.shape[0]

# 2) Define incident window (inclusive) and filter rows
start_local = tz.localize(datetime.datetime(2020,5,27,0,0,0))
end_local   = tz.localize(datetime.datetime(2020,5,27,0,30,0))
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts   = int(end_local.astimezone(pytz.UTC).timestamp())

window_df = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()
window_df['ts_dt'] = pd.to_datetime(window_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 3) Merge global thresholds and compute anomaly flags
window_with_thresh = window_df.merge(global_q, on=['cmdb_id','kpi_name'], how='left')
window_with_thresh['is_high'] = window_with_thresh['value'] >= window_with_thresh['global_P95']
window_with_thresh['is_low']  = window_with_thresh['value'] <= window_with_thresh['global_P5']

# Compute counts and earliest anomaly timestamp per pair (within window)
summary_counts = window_with_thresh.groupby(['cmdb_id','kpi_name']).agg(
    total_rows_in_window = ('value','count'),
    anomaly_count_high = ('is_high','sum'),
    anomaly_count_low  = ('is_low','sum')
).reset_index()

earliest = window_with_thresh.loc[window_with_thresh['is_high'] | window_with_thresh['is_low']].groupby(
    ['cmdb_id','kpi_name']
)['ts_dt'].min().reset_index().rename(columns={'ts_dt':'earliest_anomaly_ts'})

# Merge thresholds and earliest anomaly timestamp
summary = summary_counts.merge(global_q, on=['cmdb_id','kpi_name'], how='left').merge(
    earliest, on=['cmdb_id','kpi_name'], how='left'
)

# 4) Rank by total anomalies desc, then earliest_anomaly_timestamp asc. Limit to top 20.
summary['anomaly_sum'] = summary['anomaly_count_high'] + summary['anomaly_count_low']
summary['earliest_for_sort'] = summary['earliest_anomaly_ts'].fillna(pd.Timestamp.max.tz_localize(tz))
summary_sorted = summary.sort_values(by=['anomaly_sum','earliest_for_sort'], ascending=[False, True])

result_cols = ['cmdb_id','kpi_name','global_P95','global_P5','total_rows_in_window',
               'anomaly_count_high','anomaly_count_low','earliest_anomaly_ts']
result_df_top20 = summary_sorted[result_cols].head(20).copy()

# Format earliest_anomaly_ts: keep tz-aware timestamps, leave NaT as None for compact display
result_df_top20['earliest_anomaly_ts'] = result_df_top20['earliest_anomaly_ts'].where(
    result_df_top20['earliest_anomaly_ts'].notna(), None
)

# Final compact outputs: top 20 rows and total pairs scanned
result_df_top20, pairs_count

```
Out[2]:
```
Summary (metric.csv, incident window 2020-05-27 00:00:00+08 to 2020-05-27 00:30:00+08 inclusive):

- Total distinct (cmdb_id, kpi_name) pairs scanned: 2013.
- Produced per-pair global thresholds (P95 and P5) using the entire series, then counted anomalies in the incident window where value >= global_P95 (high) or <= global_P5 (low).
- Returned a ranked table (top 20 rows shown) with columns:
  (cmdb_id, kpi_name, global_P95, global_P5, total_rows_in_window, anomaly_count_high, anomaly_count_low, earliest_anomaly_timestamp).

Top results (highest anomaly counts, showing the leading rows from the top-20 output):
1. os_007 — Disk_rd_ios: global_P95=0.0, global_P5=0.0, rows_in_window=31, high=31, low=31, earliest=2020-05-27 00:00:00+08:00
2. os_020 — Disk_io_util: global_P95=0.0, global_P5=0.0, rows_in_window=31, high=31, low=30, earliest=2020-05-27 00:00:00+08:00
3. osb_001 — app.succee_rate: global_P95=1.0, global_P5=1.0, rows_in_window=31, high=30, low=31, earliest=2020-05-27 00:00:00+08:00
4–13. Several db_* metrics (e.g., db_001..db_013 MEM_Total and AIOS): global_P95 == global_P5, rows_in_window=30, anomaly_high=30, anomaly_low=30, earliest=2020-05-27 00:00:00+08:00
14–20. Additional metrics (e.g., db_006 MEM_real_util, os_001 Agent_ping) similarly show all or nearly-all rows flagged as anomalous with earliest anomaly at the window start.

Interpretation notes:
- Many top pairs have global_P95 == global_P5 and every or nearly every sample in the window meets both >=P95 and <=P5 (this indicates constant/identical values for those series during the global history or small variability).
- Earliest anomalies for top pairs appear at the start of the incident window (2020-05-27 00:00:00+08:00).
- The full ranked top-20 table and the total pairs scanned (2013) were produced; the output is a compact CSV-like table limited to 20 rows as requested.

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id         kpi_name  global_P95   global_P5  total_rows_in_window  anomaly_count_high  anomaly_count_low       earliest_anomaly_ts
945    os_007      Disk_rd_ios    0.000000    0.000000                    31                  31                 31 2020-05-27 00:00:00+08:00
1603   os_020     Disk_io_util    0.000000    0.000000                    31                  31                 30 2020-05-27 00:00:00+08:00
1748  osb_001  app.succee_rate    1.000000    1.000000                    31                  30                 31 2020-05-27 00:00:00+08:00
15     db_001        MEM_Total  381.902264  381.902264                    30                  30                 30 2020-05-27 00:00:00+08:00
58     db_002        MEM_Total  381.902264  381.902264                    30                  30                 30 2020-05-27 00:00:00+08:00
101    db_003        MEM_Total  381.902264  381.902264                    30                  30                 30 2020-05-27 00:00:00+08:00
187    db_005        MEM_Total  506.929929  506.929929                    30                  30                 30 2020-05-27 00:00:00+08:00
230    db_006        MEM_Total  508.252410  508.252410                    30                  30                 30 2020-05-27 00:00:00+08:00
233    db_006    MEM_real_util    0.000000    0.000000                    30                  30                 30 2020-05-27 00:00:00+08:00
273    db_007        MEM_Total  506.929929  506.929929                    30                  30                 30 2020-05-27 00:00:00+08:00
316    db_008        MEM_Total  509.244258  509.244258                    30                  30                 30 2020-05-27 00:00:00+08:00
359    db_009        MEM_Total  254.610330  254.610330                    30                  30                 30 2020-05-27 00:00:00+08:00
388    db_010             AIOS    0.000000    0.000000                    30                  30                 30 2020-05-27 00:00:00+08:00
402    db_010        MEM_Total  509.343945  509.343945                    30                  30                 30 2020-05-27 00:00:00+08:00
431    db_011             AIOS    0.000000    0.000000                    30                  30                 30 2020-05-27 00:00:00+08:00
445    db_011        MEM_Total  509.343945  509.343945                    30                  30                 30 2020-05-27 00:00:00+08:00
474    db_012             AIOS    0.000000    0.000000                    30                  30                 30 2020-05-27 00:00:00+08:00
517    db_013             AIOS    0.000000    0.000000                    30                  30                 30 2020-05-27 00:00:00+08:00
531    db_013        MEM_Total  508.997978  508.997978                    30                  30                 30 2020-05-27 00:00:00+08:00
631    os_001       Agent_ping    1.000000    1.000000                    30                  30                 30 2020-05-27 00:00:00+08:00, 2013)```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd
import pytz

# Reuse previously computed variables: summary_sorted, window_df, global_q
# If not present, raise informative error (but per session they exist)
try:
    summary_sorted
    window_df
    global_q
except NameError as e:
    raise

# Select top 50 pairs by total anomalies (anomaly_sum) from the incident-window anomaly table
top50 = summary_sorted[['cmdb_id','kpi_name','anomaly_sum','anomaly_count_high','anomaly_count_low',
                        'global_P95','global_P5','earliest_anomaly_ts']].head(50).copy()

# Aggregate stats in the incident window for these pairs
window_stats = window_df.groupby(['cmdb_id','kpi_name'])['value'].agg(
    max_value_in_window='max',
    min_value_in_window='min',
    mean_value_in_window='mean'
).reset_index()

# Merge stats into top50
top50_stats = top50.merge(window_stats, on=['cmdb_id','kpi_name'], how='left')

# Compute severities with safe division (avoid div by zero). Follow definitions:
# high_severity = if anomaly_count_high>0 then (max_value_in_window - global_P95) / max_value_in_window else 0
# low_severity  = if anomaly_count_low>0  then (global_P5 - min_value_in_window) / min_value_in_window else 0
def safe_div(numer, denom):
    return np.where((denom == 0) | pd.isna(denom), 0.0, numer / denom)

top50_stats['high_severity'] = np.where(
    top50_stats['anomaly_count_high'] > 0,
    safe_div(top50_stats['max_value_in_window'] - top50_stats['global_P95'], top50_stats['max_value_in_window']),
    0.0
)

top50_stats['low_severity'] = np.where(
    top50_stats['anomaly_count_low'] > 0,
    safe_div(top50_stats['global_P5'] - top50_stats['min_value_in_window'], top50_stats['min_value_in_window']),
    0.0
)

# Replace potential negative due to numerical precision with 0 (severity shouldn't be negative)
top50_stats['high_severity'] = top50_stats['high_severity'].clip(lower=0.0)
top50_stats['low_severity']  = top50_stats['low_severity'].clip(lower=0.0)

# Compute max severity for sorting
top50_stats['max_severity'] = top50_stats[['high_severity','low_severity']].max(axis=1)

# Sort by anomaly_sum desc then by max_severity desc
top50_sorted = top50_stats.sort_values(by=['anomaly_sum','max_severity'], ascending=[False, False])

# Prepare final columns and limit to top 20 rows for display
final_cols = ['cmdb_id','kpi_name','global_P95','global_P5','total_rows_in_window',
              'anomaly_count_high','anomaly_count_low','earliest_anomaly_ts',
              'max_value_in_window','min_value_in_window','mean_value_in_window',
              'high_severity','low_severity']

# total_rows_in_window may not be present in top50 (it was in earlier summary_counts); try to bring it from summary if exists
if 'total_rows_in_window' not in top50_sorted.columns:
    # Attempt to merge from summary (which exists as 'summary' variable)
    try:
        top50_sorted = top50_sorted.merge(summary[['cmdb_id','kpi_name','total_rows_in_window']],
                                          on=['cmdb_id','kpi_name'], how='left')
    except Exception:
        top50_sorted['total_rows_in_window'] = np.nan

# Ensure column order exists
display_df = top50_sorted[final_cols].head(20).copy()

# Compact summary counts by component prefix among the top-50 pairs
top50_pairs = top50[['cmdb_id','kpi_name']].copy()
count_os = top50_pairs['cmdb_id'].str.startswith('os_').sum()
count_docker = top50_pairs['cmdb_id'].str.startswith('docker_').sum()
count_db = top50_pairs['cmdb_id'].str.startswith('db_').sum()
count_other = top50.shape[0] - (count_os + count_docker + count_db)

prefix_counts = pd.Series({
    'count_os': int(count_os),
    'count_docker': int(count_docker),
    'count_db': int(count_db),
    'count_other': int(count_other)
})

# Display compact results: table (top 20 rows) and prefix counts
display_df, prefix_counts

```
Out[3]:
```
Summary of the requested analysis (top-50 pairs by total anomalies; showing top 20 rows):

What I computed
- Selected the top 50 (cmdb_id, kpi_name) pairs by total anomalies (anomaly_count_high + anomaly_count_low) from the incident window (2020-05-27 00:00:00+08 to 00:30:00+08).
- For those pairs, using only rows in the incident window, I reported: global_P95, global_P5 (computed globally across the whole series), total_rows_in_window, anomaly_count_high, anomaly_count_low, earliest_anomaly_timestamp, max/min/mean in-window values, and computed high_severity and low_severity per your formulas.
- Returned the results sorted by total anomalies desc then by max(high_severity, low_severity) desc; limited display to top 20 rows.

Top observations (top rows from the returned table)
1. os_007 — Disk_rd_ios
   - global_P95=0.0, global_P5=0.0, rows_in_window=31, high=31, low=31, earliest=2020-05-27 00:00:00+08
   - max=min=mean=0.0 → high_severity=0.0, low_severity=0.0
   - anomaly_sum = 62 (all samples flagged both high and low due to thresholds)

2. os_020 — Disk_io_util
   - global_P95=0.0, global_P5=0.0, rows_in_window=31, high=31, low=30, earliest=2020-05-27 00:00:00+08
   - max_in_window=0.333333, min=0.0, mean≈0.01075 → high_severity=1.0, low_severity=0.0
   - anomaly_sum = 61, high_severity is 1.0 (because global_P95==0 and max>0)

3. osb_001 — app.succee_rate
   - global_P95=1.0, global_P5=1.0, rows_in_window=31, high=30, low=31, earliest=2020-05-27 00:00:00+08
   - max=1.0, min≈0.9385, mean≈0.9980 → high_severity=0.0, low_severity≈0.0655
   - anomaly_sum = 61

4–19. Many db_* metrics (e.g., db_001..db_013 MEM_Total, several AIOS and MEM_real_util entries) and os_001 Agent_ping appear in the top-20:
   - For multiple db_* MEM_Total entries: global_P95 == global_P5 == in-window constant value (e.g., 381.902264, 506.929929, 509.343945, etc.), rows_in_window=30, anomaly_high=30, anomaly_low=30, earliest=window start, and severities are 0.0 because max==min==global thresholds.
   - Several metrics have anomaly_count_high and anomaly_count_low equal to the window row count (indicating little or no variance or thresholds equal to observed values).

Severity notes
- Many top pairs show high/low severity 0.0 because the series are effectively constant (global thresholds equal to in-window values), causing anomalies to be flagged but severity to be zero.
- os_020 (Disk_io_util) shows a high_severity of 1.0 because global_P95 is 0 while the in-window max is >0 (makes normalized severity = 1).
- osb_001 (app.succee_rate) shows a modest low_severity (~0.0655) due to min < global_P5.

Component-level counts among the top 50 pairs
- count_os: 33
- count_docker: 0
- count_db: 16
- count_other: 1

Compact conclusion
- The top anomalous pairs are dominated by OS-level metrics (33 of 50) and many DB MEM_Total metrics; most show anomalies in essentially all samples in the window but often with zero severity because thresholds equal observed values. A few pairs (e.g., os_020 Disk_io_util) show non-zero severity indicating a measurable deviation from the global P95/P5 thresholds.

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id         kpi_name  global_P95   global_P5  total_rows_in_window  anomaly_count_high  anomaly_count_low       earliest_anomaly_ts  max_value_in_window  min_value_in_window  mean_value_in_window  high_severity  low_severity
0    os_007      Disk_rd_ios    0.000000    0.000000                    31                  31                 31 2020-05-27 00:00:00+08:00             0.000000             0.000000              0.000000            0.0       0.00000
1    os_020     Disk_io_util    0.000000    0.000000                    31                  31                 30 2020-05-27 00:00:00+08:00             0.333333             0.000000              0.010753            1.0       0.00000
2   osb_001  app.succee_rate    1.000000    1.000000                    31                  30                 31 2020-05-27 00:00:00+08:00             1.000000             0.938500              0.998016            0.0       0.06553
3    db_001        MEM_Total  381.902264  381.902264                    30                  30                 30 2020-05-27 00:00:00+08:00           381.902264           381.902264            381.902264            0.0       0.00000
4    db_002        MEM_Total  381.902264  381.902264                    30                  30                 30 2020-05-27 00:00:00+08:00           381.902264           381.902264            381.902264            0.0       0.00000
5    db_003        MEM_Total  381.902264  381.902264                    30                  30                 30 2020-05-27 00:00:00+08:00           381.902264           381.902264            381.902264            0.0       0.00000
6    db_005        MEM_Total  506.929929  506.929929                    30                  30                 30 2020-05-27 00:00:00+08:00           506.929929           506.929929            506.929929            0.0       0.00000
7    db_006        MEM_Total  508.252410  508.252410                    30                  30                 30 2020-05-27 00:00:00+08:00           508.252410           508.252410            508.252410            0.0       0.00000
8    db_006    MEM_real_util    0.000000    0.000000                    30                  30                 30 2020-05-27 00:00:00+08:00             0.000000             0.000000              0.000000            0.0       0.00000
9    db_007        MEM_Total  506.929929  506.929929                    30                  30                 30 2020-05-27 00:00:00+08:00           506.929929           506.929929            506.929929            0.0       0.00000
10   db_008        MEM_Total  509.244258  509.244258                    30                  30                 30 2020-05-27 00:00:00+08:00           509.244258           509.244258            509.244258            0.0       0.00000
11   db_009        MEM_Total  254.610330  254.610330                    30                  30                 30 2020-05-27 00:00:00+08:00           254.610330           254.610330            254.610330            0.0       0.00000
12   db_010             AIOS    0.000000    0.000000                    30                  30                 30 2020-05-27 00:00:00+08:00             0.000000             0.000000              0.000000            0.0       0.00000
13   db_010        MEM_Total  509.343945  509.343945                    30                  30                 30 2020-05-27 00:00:00+08:00           509.343945           509.343945            509.343945            0.0       0.00000
14   db_011             AIOS    0.000000    0.000000                    30                  30                 30 2020-05-27 00:00:00+08:00             0.000000             0.000000              0.000000            0.0       0.00000
15   db_011        MEM_Total  509.343945  509.343945                    30                  30                 30 2020-05-27 00:00:00+08:00           509.343945           509.343945            509.343945            0.0       0.00000
16   db_012             AIOS    0.000000    0.000000                    30                  30                 30 2020-05-27 00:00:00+08:00             0.000000             0.000000              0.000000            0.0       0.00000
17   db_013             AIOS    0.000000    0.000000                    30                  30                 30 2020-05-27 00:00:00+08:00             0.000000             0.000000              0.000000            0.0       0.00000
18   db_013        MEM_Total  508.997978  508.997978                    30                  30                 30 2020-05-27 00:00:00+08:00           508.997978           508.997978            508.997978            0.0       0.00000
19   os_001       Agent_ping    1.000000    1.000000                    30                  30                 30 2020-05-27 00:00:00+08:00             1.000000             1.000000              1.000000            0.0       0.00000, count_os        33
count_docker     0
count_db        16
count_other      1
dtype: int64)```
```

In [ ]:
# In[4]:

# Reuse in-kernel variables from previous steps:
# top50_sorted contains the top-50 pairs with computed fields including:
# ['cmdb_id','kpi_name','global_P95','global_P5','total_rows_in_window',
#  'anomaly_count_high','anomaly_count_low','earliest_anomaly_ts',
#  'max_value_in_window','min_value_in_window','mean_value_in_window',
#  'high_severity','low_severity','max_severity','anomaly_sum', ...]

import pandas as pd

# Ensure variable exists
try:
    top50_sorted
except NameError:
    raise RuntimeError("Expected variable 'top50_sorted' not found in kernel state.")

# Filter pairs where max(high_severity, low_severity) > 0 and anomaly_sum >= 1
cond = (top50_sorted[['high_severity','low_severity']].max(axis=1) > 0.0) & (top50_sorted['anomaly_sum'] >= 1)
filtered = top50_sorted.loc[cond].copy()

# Sort by max severity desc, then earliest_anomaly_ts asc
filtered['max_severity'] = filtered[['high_severity','low_severity']].max(axis=1)
# Replace NaT with large timestamp for stable sorting if needed
filtered['earliest_for_sort'] = filtered['earliest_anomaly_ts'].fillna(pd.Timestamp.max.tz_localize('Asia/Shanghai'))

filtered_sorted = filtered.sort_values(by=['max_severity','earliest_for_sort'], ascending=[False, True])

# Select and order the requested columns
out_cols = ['cmdb_id','kpi_name','global_P95','global_P5','total_rows_in_window',
            'anomaly_count_high','anomaly_count_low','earliest_anomaly_ts',
            'max_value_in_window','min_value_in_window','mean_value_in_window',
            'high_severity','low_severity']

result_df = filtered_sorted[out_cols].head(20).copy()

# Compact component-level prefix counts among the matching pairs
matching_pairs = filtered_sorted[['cmdb_id','kpi_name']].head(10000)  # safe slice
count_os = int(matching_pairs['cmdb_id'].str.startswith('os_').sum())
count_docker = int(matching_pairs['cmdb_id'].str.startswith('docker_').sum())
count_db = int(matching_pairs['cmdb_id'].str.startswith('db_').sum())
count_other = int(matching_pairs.shape[0] - (count_os + count_docker + count_db))

prefix_counts = pd.Series({
    'count_os': count_os,
    'count_docker': count_docker,
    'count_db': count_db,
    'count_other': count_other
})

# Display compact CSV-like table (top 20) and prefix counts
result_df, prefix_counts

```
Out[4]:
```
summary = (
    "Filtered incident-window results (condition: max(high_severity,low_severity) > 0 and anomalies >= 1):\n\n"
    "Total matching pairs: 2\n\n"
    "Rows (top 20 shown):\n"
    "1) cmdb_id=os_020, kpi_name=Disk_io_util\n"
    "   - global_P95=0.0, global_P5=0.0\n"
    "   - total_rows_in_window=31, anomaly_count_high=31, anomaly_count_low=30\n"
    "   - earliest_anomaly_timestamp=2020-05-27 00:00:00+08:00\n"
    "   - max=0.333333, min=0.000000, mean≈0.010753\n"
    "   - high_severity=1.0, low_severity=0.0\n"
    "   - Interpretation: global thresholds are zero while in-window max > 0, giving a high_severity of 1.0 (strong positive deviation).\n\n"
    "2) cmdb_id=osb_001, kpi_name=app.succee_rate\n"
    "   - global_P95=1.0, global_P5=1.0\n"
    "   - total_rows_in_window=31, anomaly_count_high=30, anomaly_count_low=31\n"
    "   - earliest_anomaly_timestamp=2020-05-27 00:00:00+08:00\n"
    "   - max=1.000000, min≈0.938500, mean≈0.998016\n"
    "   - high_severity=0.0, low_severity≈0.06553\n"
    "   - Interpretation: values are very close to 1.0 but occasionally dip below, producing a modest low_severity (~6.6%).\n\n"
    "Component-level counts among matches:\n"
    "- count_os: 1\n"
    "- count_docker: 0\n"
    "- count_db: 0\n"
    "- count_other: 1\n\n"
    "Overall: only two pairs met the filter. os_020/Disk_io_util shows the clearest severity (high_severity=1.0). osb_001/app.succee_rate shows a small low-side severity."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id         kpi_name  global_P95  global_P5  total_rows_in_window  anomaly_count_high  anomaly_count_low       earliest_anomaly_ts  max_value_in_window  min_value_in_window  mean_value_in_window  high_severity  low_severity
1   os_020     Disk_io_util         0.0        0.0                    31                  31                 30 2020-05-27 00:00:00+08:00             0.333333               0.0000              0.010753            1.0       0.00000
2  osb_001  app.succee_rate         1.0        1.0                    31                  30                 31 2020-05-27 00:00:00+08:00             1.000000               0.9385              0.998016            0.0       0.06553, count_os        1
count_docker    0
count_db        0
count_other     1
dtype: int64)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz
import datetime

# Reuse trace_df if present; otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

# Ensure value numeric if needed
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Timezone setup (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(datetime.datetime(2020,5,27,0,0,0))
end_local   = tz.localize(datetime.datetime(2020,5,27,0,30,0))
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts   = int(end_local.astimezone(pytz.UTC).timestamp())

# Filter to incident window (inclusive)
window_trace = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] <= end_ts)].copy()
window_trace['ts_dt'] = pd.to_datetime(window_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) For cmdb_id in {'os_020','osb_001'}
cmdb_targets = ['os_020','osb_001']
mask1 = window_trace['cmdb_id'].isin(cmdb_targets)
group1 = (window_trace[mask1]
          .groupby('cmdb_id', dropna=False)
          .agg(
              count_of_rows = ('timestamp','count'),
              distinct_trace_names = ('trace_name', lambda s: sorted(pd.unique(s)) ),
              earliest_ts = ('ts_dt','min'),
              latest_ts = ('ts_dt','max')
          ).reset_index())

# Limit distinct trace names to 20 and convert to compact string for display
def compact_list(lst, limit=20):
    if not isinstance(lst, list):
        return ""
    if len(lst) > limit:
        return ", ".join(lst[:limit]) + f", ...(+{len(lst)-limit} more)"
    return ", ".join(lst)

group1['distinct_trace_names'] = group1['distinct_trace_names'].apply(lambda x: compact_list(x, 20))

# Up to 10 sample rows for these cmdb_ids
sample1 = window_trace[mask1][['ts_dt','cmdb_id','trace_name','value']].head(10).copy()
sample1 = sample1.rename(columns={'ts_dt':'timestamp'})

# 2) Rows where trace_name contains substring 'os_020' or 'osb_001' anywhere
pattern = r'os_020|osb_001'
mask2 = window_trace['trace_name'].str.contains(pattern, na=False)
matched2 = window_trace[mask2].copy()

group2 = (matched2
          .groupby('trace_name', dropna=False)
          .agg(
              count_of_rows = ('timestamp','count'),
              earliest_ts = ('ts_dt','min'),
              latest_ts = ('ts_dt','max')
          ).reset_index()
          .sort_values('count_of_rows', ascending=False)
          .head(20)
         )

# Up to 10 sample rows matching these substrings
sample2 = matched2[['ts_dt','cmdb_id','trace_name','value']].head(10).copy()
sample2 = sample2.rename(columns={'ts_dt':'timestamp'})

# Compact outputs (CSV-like summaries and sample rows)
group1_summary = group1[['cmdb_id','count_of_rows','distinct_trace_names','earliest_ts','latest_ts']]
group2_summary = group2[['trace_name','count_of_rows','earliest_ts','latest_ts']]

# Display results (variables)
group1_summary, sample1, group2_summary, sample2

```
Out[5]:
```
Summary of trace.csv scan for incident window 2020-05-27 00:00:00+08 to 2020-05-27 00:30:00+08 (inclusive):

- Filter by cmdb_id in {'os_020', 'osb_001'}:
  - count_of_rows: 0 (no rows found)
  - distinct trace_name values: none
  - earliest_timestamp / latest_timestamp: none
  - sample rows: none

- Filter by trace_name containing substring 'os_020' or 'osb_001':
  - distinct trace_name entries found: 0
  - total matching rows: 0
  - earliest_timestamp / latest_timestamp: none
  - sample rows: none

Conclusion: Within the specified incident window there are no trace.csv rows for cmdb_id os_020 or osb_001, nor any trace_name containing those substrings.

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [cmdb_id, count_of_rows, distinct_trace_names, earliest_ts, latest_ts]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, trace_name, value]
Index: [], Empty DataFrame
Columns: [trace_name, count_of_rows, earliest_ts, latest_ts]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, trace_name, value]
Index: [])```
```

In [ ]:
# In[6]:

import numpy as np
import pandas as pd
import pytz
import datetime

# Reuse metric_df and tz if available, otherwise load/define
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure numeric 'value'
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Focus cmdb_id
target_cmdb = 'os_020'

# 1) Compute global P95 and P5 per (cmdb_id, kpi_name) using full series (must be done BEFORE filtering)
global_thresh = metric_df.groupby(['cmdb_id','kpi_name'], dropna=False)['value'].agg(
    global_P5=lambda x: x.quantile(0.05),
    global_P95=lambda x: x.quantile(0.95)
).reset_index()

# Keep only thresholds for os_020
os020_thresh = global_thresh[global_thresh['cmdb_id'] == target_cmdb].copy()

# 2) Define incident window (inclusive) in UTC timestamps
start_local = tz.localize(datetime.datetime(2020,5,27,0,0,0))
end_local   = tz.localize(datetime.datetime(2020,5,27,0,30,0))
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts   = int(end_local.astimezone(pytz.UTC).timestamp())

# Filter metric_df to os_020 and window
os020_window = metric_df[(metric_df['cmdb_id'] == target_cmdb) &
                         (metric_df['timestamp'] >= start_ts) &
                         (metric_df['timestamp'] <= end_ts)].copy()

# Convert timestamp to Asia/Shanghai tz-aware datetime for display
os020_window['ts_dt'] = pd.to_datetime(os020_window['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# If no kpis present, return empty results and count 0
if os020_window.shape[0] == 0:
    result_df = pd.DataFrame(columns=[
        'kpi_name','global_P95','global_P5','total_rows_in_window',
        'anomaly_count_high','anomaly_count_low','earliest_anomaly_timestamp',
        'max_value_in_window','min_value_in_window','mean_value_in_window',
        'high_severity','low_severity'
    ])
    kpi_count = 0
else:
    # 3) For each kpi_name present in the window, compute required stats
    # Merge global thresholds into window rows
    os020_window = os020_window.merge(os020_thresh[['cmdb_id','kpi_name','global_P95','global_P5']],
                                      on=['cmdb_id','kpi_name'], how='left')

    # Flags
    os020_window['is_high'] = os020_window['value'] >= os020_window['global_P95']
    os020_window['is_low']  = os020_window['value'] <= os020_window['global_P5']

    # Aggregate per kpi_name
    agg_stats = os020_window.groupby('kpi_name').agg(
        global_P95 = ('global_P95','first'),
        global_P5  = ('global_P5','first'),
        total_rows_in_window = ('value','count'),
        anomaly_count_high = ('is_high','sum'),
        anomaly_count_low  = ('is_low','sum'),
        max_value_in_window = ('value','max'),
        min_value_in_window = ('value','min'),
        mean_value_in_window = ('value','mean')
    ).reset_index()

    # Earliest anomaly timestamp per kpi_name where either condition holds
    anomalies = os020_window[os020_window['is_high'] | os020_window['is_low']].copy()
    earliest = anomalies.groupby('kpi_name')['ts_dt'].min().reset_index().rename(columns={'ts_dt':'earliest_anomaly_timestamp'})
    agg_stats = agg_stats.merge(earliest, on='kpi_name', how='left')

    # Compute severities with safe division and rules
    def safe_div(numer, denom):
        return np.where((denom == 0) | pd.isna(denom), 0.0, numer / denom)

    agg_stats['high_severity'] = np.where(
        agg_stats['anomaly_count_high'] > 0,
        safe_div(agg_stats['max_value_in_window'] - agg_stats['global_P95'], agg_stats['max_value_in_window']),
        0.0
    )

    agg_stats['low_severity'] = np.where(
        agg_stats['anomaly_count_low'] > 0,
        safe_div(agg_stats['global_P5'] - agg_stats['min_value_in_window'], agg_stats['min_value_in_window']),
        0.0
    )

    # Clip negatives to 0
    agg_stats['high_severity'] = agg_stats['high_severity'].clip(lower=0.0)
    agg_stats['low_severity']  = agg_stats['low_severity'].clip(lower=0.0)

    # Sorting: by max(high_severity, low_severity) desc then by anomaly_sum desc
    agg_stats['max_severity'] = agg_stats[['high_severity','low_severity']].max(axis=1)
    agg_stats['anomaly_sum'] = agg_stats['anomaly_count_high'] + agg_stats['anomaly_count_low']
    agg_stats_sorted = agg_stats.sort_values(by=['max_severity','anomaly_sum'], ascending=[False, False])

    # Prepare final columns and limit to top 20
    cols = ['kpi_name','global_P95','global_P5','total_rows_in_window',
            'anomaly_count_high','anomaly_count_low','earliest_anomaly_timestamp',
            'max_value_in_window','min_value_in_window','mean_value_in_window',
            'high_severity','low_severity']
    result_df = agg_stats_sorted[cols].head(20).copy()

    # Count distinct kpi_name entries for os_020 scanned
    kpi_count = agg_stats.shape[0]

# Display compact results: result_df and kpi_count
result_df, kpi_count

```
Out[6]:
```
Summary of os_020 KPI scan (incident window 2020-05-27 00:00:00+08 to 00:30:00+08):

- Total distinct kpi_name entries for os_020 scanned in the window: 50.

- What was computed per kpi:
  - global P95 and P5 (computed from the full series, before window filtering),
  - counts and anomaly counts in the incident window (value >= global_P95 as high, <= global_P5 as low),
  - earliest anomaly time in-window (if any),
  - in-window max/min/mean, and high_severity / low_severity as defined.

- Top results (sorted by max(high_severity, low_severity) then by anomaly count). I show the top 20 KPI rows (compact highlights):

1) Disk_io_util
   - global_P95=0.0, global_P5=0.0; rows_in_window=31; high=31, low=30
   - earliest anomaly: 2020-05-27 00:00:00+08
   - max=0.333333, min=0.0, mean≈0.010753
   - high_severity=1.000, low_severity=0.000
   - Note: strongest positive deviation (global_P95==0 but max>0).

2) Disk_svctm
   - global_P95=2.2125, global_P5=0.0; rows=30; high=2, low=25
   - earliest: 2020-05-27 00:00:00+08
   - max=29.78, min=0.0, mean≈1.201
   - high_severity≈0.9257, low_severity=0.0

3) Processor_load_1_min
   - global_P95≈0.043667, global_P5≈0.00875; rows=29; high=2, low=2
   - earliest: 2020-05-27 00:07:00+08
   - max≈0.05625, min≈0.005833, mean≈0.02300
   - high_severity≈0.2237, low_severity≈0.5001

4) Page_po
   - global_P95=306.8, global_P5=52.1; rows=30; high=2, low=2
   - earliest: 2020-05-27 00:20:00+08
   - max=407.0, min=35.0, mean=238.0
   - high_severity≈0.2462, low_severity≈0.4886

5) CPU_iowait_time
   - global_P95≈0.03247, global_P5≈0.001009; rows=30; high=2, low=2
   - earliest: 2020-05-27 00:10:00+08
   - max≈0.039667, min≈0.000696, mean≈0.008769
   - high_severity≈0.1815, low_severity≈0.4494

6) Disk_wr_kbs
   - global_P95≈0.38634, global_P5≈0.0339; rows=30; high=2, low=2
   - earliest: 2020-05-27 00:00:00+08
   - max≈0.418419, min≈0.024133, mean≈0.230589
   - high_severity≈0.0767, low_severity≈0.4047

7) Sent_packets
   - global_P95≈65.416667, global_P5≈24.333333; rows=6; high=1, low=1
   - earliest: 2020-05-27 00:00:00+08
   - max≈65.666667, min≈17.333333, mean≈51.222222
   - high_severity≈0.0038, low_severity≈0.4038

8) Received_packets
   - global_P95=57.0, global_P5=21.5; rows=6; high=2, low=1
   - earliest: 2020-05-27 00:00:00+08
   - max=57.0, min≈15.333333, mean≈45.111111
   - high_severity=0.0, low_severity≈0.4022

9) Send_total
   - global_P95≈0.23078, global_P5≈0.026636; rows=30; high=2, low=2
   - earliest: 2020-05-27 00:20:00+08
   - max≈0.282759, min≈0.019786, mean≈0.170975
   - high_severity≈0.1838, low_severity≈0.3462

10) Disk_avgqu_sz
    - global_P95≈0.038128, global_P5≈0.004208; rows=29; high=2, low=2
    - earliest: 2020-05-27 00:10:00+08
    - max≈0.050096, min≈0.003982, mean≈0.020049
    - high_severity≈0.2389, low_severity≈0.0568

11) Recv_total
    - global_P95≈237.8051, global_P5≈26.33321; rows=30; high=2, low=2
    - earliest: 2020-05-27 00:20:00+08
    - max≈286.0064, min≈21.3025, mean≈175.90835
    - high_severity≈0.1685, low_severity≈0.2362

12) Outgoing_network_traffic
    - global_P95≈0.048501, global_P5≈0.005367; rows=30; high=2, low=2
    - earliest: 2020-05-27 00:02:00+08
    - max≈0.063131, min=0.0, mean≈0.032825
    - high_severity≈0.2317, low_severity=0.0

13) Incoming_network_traffic
    - global_P95≈0.026803, global_P5≈0.003281; rows=30; high=2, low=2
    - earliest: 2020-05-27 00:23:00+08
    - max≈0.032445, min≈0.002696, mean≈0.020101
    - high_severity≈0.1739, low_severity≈0.2169

14) Disk_wr_ios
    - global_P95=2.55, global_P5=1.0; rows=30; high=2, low=6
    - earliest: 2020-05-27 00:01:00+08
    - max=3.0, min=1.0, mean≈1.866667
    - high_severity=0.15, low_severity=0.0

15) System_wait_queue_length
    - global_P95=1.75, global_P5=0.0; rows=6; high=1, low=4
    - earliest: 2020-05-27 00:09:00+08
    - max=2.0, min=0.0, mean=0.5
    - high_severity=0.125, low_severity=0.0

16) CPU_user_time
    - global_P95≈2.185104, global_P5≈1.552822; rows=30; high=2, low=2
    - earliest: 2020-05-27 00:18:00+08
    - max≈2.425458, min≈1.483516, mean≈1.979627
    - high_severity≈0.0991, low_severity≈0.0467

17) CPU_util_pct
    - global_P95≈3.323637, global_P5≈2.611766; rows=24; high=3, low=2
    - earliest: 2020-05-27 00:10:00+08
    - max≈3.624112, min≈2.533354, mean≈3.100334
    - high_severity≈0.0829, low_severity≈0.0310

18) CPU_system_time
    - global_P95≈1.10223, global_P5≈0.941972; rows=30; high=2, low=2
    - earliest: 2020-05-27 00:20:00+08
    - max≈1.118722, min≈0.885362, mean≈1.031762
    - high_severity≈0.0147, low_severity≈0.0639

19) Disk_await
    - global_P95=2.09, global_P5=0.0; rows=29; high=2, low=25
    - earliest: 2020-05-27 00:00:00+08
    - max≈2.22, min=0.0, mean≈0.254138
    - high_severity≈0.0586, low_severity=0.0

20) Processor_load_5_min
    - global_P95≈0.031166, global_P5≈0.01625; rows=29; high=2, low=3
    - earliest: 2020-05-27 00:01:00+08
    - max≈0.0325, min≈0.015833, mean≈0.022435
    - high_severity≈0.0410, low_severity≈0.0263

Key takeaways
- 50 KPIs for os_020 were present in the incident window.
- The largest positive severity is Disk_io_util (high_severity = 1.0) and Disk_svctm also shows a very large high_severity (~0.926) — both indicate strong positive deviations above the global P95.
- Several KPIs show notable low-side severity (e.g., Processor_load_1_min low_severity ≈0.50, Page_po low_severity ≈0.49), meaning their in-window minima fell substantially below global P5.
- Many KPIs have only a few anomaly points in-window (anomaly counts often 1–3) while a few have anomalies across most samples (e.g., Disk_io_util).
- Earliest anomalies often occur at or near the window start (00:00:00+08), but some KPIs first anomalous timestamps are later (e.g., Processor_load_1_min at 00:07, Page_po at 00:20).

If you want, I can output the compact CSV-like table (top 20 rows) exactly as values shown.

The original code execution output of IPython Kernel is also provided below for reference:

(                    kpi_name  global_P95  global_P5  total_rows_in_window  anomaly_count_high  anomaly_count_low earliest_anomaly_timestamp  max_value_in_window  min_value_in_window  mean_value_in_window  high_severity  low_severity
10              Disk_io_util    0.000000   0.000000                    31                  31                 30  2020-05-27 00:00:00+08:00             0.333333             0.000000              0.010753       1.000000      0.000000
13                Disk_svctm    2.212500   0.000000                    30                   2                 25  2020-05-27 00:00:00+08:00            29.780000             0.000000              1.201000       0.925705      0.000000
34      Processor_load_1_min    0.043667   0.008750                    29                   2                  2  2020-05-27 00:07:00+08:00             0.056250             0.005833              0.023003       0.223698      0.500120
32                   Page_po  306.800000  52.100000                    30                   2                  2  2020-05-27 00:20:00+08:00           407.000000            35.000000            238.000000       0.246192      0.488571
3            CPU_iowait_time    0.032466   0.001009                    30                   2                  2  2020-05-27 00:10:00+08:00             0.039667             0.000696              0.008769       0.181534      0.449353
15               Disk_wr_kbs    0.386339   0.033900                    30                   2                  2  2020-05-27 00:00:00+08:00             0.418419             0.024133              0.230589       0.076670      0.404726
42              Sent_packets   65.416667  24.333333                     6                   1                  1  2020-05-27 00:00:00+08:00            65.666667            17.333333             51.222222       0.003807      0.403846
37          Received_packets   57.000000  21.500000                     6                   2                  1  2020-05-27 00:00:00+08:00            57.000000            15.333333             45.111111       0.000000      0.402174
40                Send_total    0.230782   0.026636                    30                   2                  2  2020-05-27 00:20:00+08:00             0.282759             0.019786              0.170975       0.183822      0.346199
8              Disk_avgqu_sz    0.038128   0.004208                    29                   2                  2  2020-05-27 00:10:00+08:00             0.050096             0.003982              0.020049       0.238905      0.056755
39                Recv_total  237.805078  26.333210                    30                   2                  2  2020-05-27 00:20:00+08:00           286.006353            21.302487            175.908350       0.168532      0.236157
30  Outgoing_network_traffic    0.048501   0.005367                    30                   2                  2  2020-05-27 00:02:00+08:00             0.063131             0.000000              0.032825       0.231733      0.000000
22  Incoming_network_traffic    0.026803   0.003281                    30                   2                  2  2020-05-27 00:23:00+08:00             0.032445             0.002696              0.020101       0.173889      0.216926
14               Disk_wr_ios    2.550000   1.000000                    30                   2                  6  2020-05-27 00:01:00+08:00             3.000000             1.000000              1.866667       0.150000      0.000000
47  System_wait_queue_length    1.750000   0.000000                     6                   1                  4  2020-05-27 00:09:00+08:00             2.000000             0.000000              0.500000       0.125000      0.000000
5              CPU_user_time    2.185104   1.552822                    30                   2                  2  2020-05-27 00:18:00+08:00             2.425458             1.483516              1.979627       0.099096      0.046717
6               CPU_util_pct    3.323637   2.611766                    24                   3                  2  2020-05-27 00:10:00+08:00             3.624112             2.533354              3.100334       0.082910      0.030952
4            CPU_system_time    1.102230   0.941972                    30                   2                  2  2020-05-27 00:20:00+08:00             1.118722             0.885362              1.031762       0.014742      0.063940
9                 Disk_await    2.090000   0.000000                    29                   2                 25  2020-05-27 00:00:00+08:00             2.220000             0.000000              0.254138       0.058559      0.000000
35      Processor_load_5_min    0.031166   0.016250                    29                   2                  3  2020-05-27 00:01:00+08:00             0.032500             0.015833              0.022435       0.041034      0.026337, 50)```
```

In [ ]:
# In[7]:

import numpy as np
import pandas as pd
import pytz
import datetime

# Reuse metric_df and tz if available
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure numeric
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

target_cmdb = 'os_020'
kpis_of_interest = ['Disk_io_util', 'Disk_svctm']

# 1) Compute global P95 and P5 for the target KPIs using full series (before filtering)
global_thresh = (metric_df[metric_df['cmdb_id'] == target_cmdb]
                 .groupby('kpi_name', dropna=False)['value']
                 .agg(global_P5=lambda x: x.quantile(0.05),
                      global_P95=lambda x: x.quantile(0.95))
                 .reset_index())

# Keep only the two KPIs
global_thresh = global_thresh[global_thresh['kpi_name'].isin(kpis_of_interest)].set_index('kpi_name')

# Incident window timestamps (inclusive)
start_local = tz.localize(datetime.datetime(2020,5,27,0,0,0))
end_local   = tz.localize(datetime.datetime(2020,5,27,0,30,0))
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts   = int(end_local.astimezone(pytz.UTC).timestamp())

# Filter data to os_020 within window
os020_window = metric_df[(metric_df['cmdb_id'] == target_cmdb) &
                         (metric_df['timestamp'] >= start_ts) &
                         (metric_df['timestamp'] <= end_ts)].copy()

# Convert ts to tz-aware datetimes for display
os020_window['ts_dt'] = pd.to_datetime(os020_window['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Helper safe division
def safe_div(numer, denom):
    return np.where((denom == 0) | pd.isna(denom), 0.0, numer / denom)

# Function to detect consecutive-minute segments where value >= global_P95 for a KPI
def detect_segments(df_kpi, global_p95):
    if df_kpi.empty:
        return pd.DataFrame(columns=[
            'kpi_name','segment_start_timestamp','segment_end_timestamp','length_in_minutes',
            'max_value_in_segment','min_value_in_segment','mean_value_in_segment','global_P95','max_severity'
        ])
    # Filter anomaly points (value >= global_P95)
    anomalous = df_kpi[df_kpi['value'] >= global_p95].sort_values('timestamp').copy()
    if anomalous.empty:
        return pd.DataFrame(columns=[
            'kpi_name','segment_start_timestamp','segment_end_timestamp','length_in_minutes',
            'max_value_in_segment','min_value_in_segment','mean_value_in_segment','global_P95','max_severity'
        ])
    # Identify consecutive groups: adjacent minute timestamps differ by exactly 60 seconds
    anomalous['ts_diff'] = anomalous['timestamp'].diff().fillna(0)
    anomalous['new_segment'] = (anomalous['ts_diff'] != 60).cumsum()
    segs = anomalous.groupby('new_segment').agg(
        kpi_name = ('kpi_name','first'),
        segment_start_ts = ('timestamp','min'),
        segment_end_ts = ('timestamp','max'),
        length_in_minutes = ('timestamp','count'),
        max_value_in_segment = ('value','max'),
        min_value_in_segment = ('value','min'),
        mean_value_in_segment = ('value','mean')
    ).reset_index(drop=True)
    # Convert timestamps to tz-aware datetimes
    segs['segment_start_timestamp'] = pd.to_datetime(segs['segment_start_ts'], unit='s', utc=True).dt.tz_convert(tz)
    segs['segment_end_timestamp']   = pd.to_datetime(segs['segment_end_ts'], unit='s', utc=True).dt.tz_convert(tz)
    segs['global_P95'] = global_p95
    # Compute max_severity safely
    segs['max_severity'] = safe_div(segs['max_value_in_segment'] - segs['global_P95'], segs['max_value_in_segment'])
    segs['max_severity'] = segs['max_severity'].clip(lower=0.0)
    # Select and order columns
    segs = segs[['kpi_name','segment_start_timestamp','segment_end_timestamp','length_in_minutes',
                 'max_value_in_segment','min_value_in_segment','mean_value_in_segment','global_P95','max_severity']]
    return segs

# Detect segments for each KPI and collect results
segments_list = []
for kpi in kpis_of_interest:
    # get global P95 (if missing, skip)
    if kpi not in global_thresh.index:
        continue
    g95 = float(global_thresh.loc[kpi,'global_P95'])
    df_kpi = os020_window[os020_window['kpi_name'] == kpi].copy()
    segs = detect_segments(df_kpi, g95)
    segments_list.append(segs)

segments_df = pd.concat(segments_list, ignore_index=True) if segments_list else pd.DataFrame()

# 2) Identify earliest segment start timestamp across the two KPIs
if segments_df.shape[0] > 0:
    earliest_disk_anomaly_ts = segments_df['segment_start_timestamp'].min()
else:
    earliest_disk_anomaly_ts = None

# 3) For that earliest timestamp, list all os_020 KPIs anomalous at that exact timestamp
if earliest_disk_anomaly_ts is None:
    anomalies_at_earliest_df = pd.DataFrame(columns=[
        'kpi_name','value_at_timestamp','global_P95','global_P5','anomaly_type','severity'
    ])
else:
    # Convert earliest timestamp to epoch seconds UTC for matching metric_df.timestamp
    earliest_utc = earliest_disk_anomaly_ts.tz_convert(pytz.UTC)
    earliest_epoch = int(earliest_utc.timestamp())
    # Get all os_020 rows at that exact timestamp
    rows_at_ts = metric_df[(metric_df['cmdb_id'] == target_cmdb) & (metric_df['timestamp'] == earliest_epoch)].copy()
    if rows_at_ts.empty:
        anomalies_at_earliest_df = pd.DataFrame(columns=[
            'kpi_name','value_at_timestamp','global_P95','global_P5','anomaly_type','severity'
        ])
    else:
        # Merge global thresholds (P95,P5) for os_020 KPIs (computed globally)
        os020_global = metric_df[metric_df['cmdb_id'] == target_cmdb].groupby('kpi_name')['value'].agg(
            global_P5=lambda x: x.quantile(0.05),
            global_P95=lambda x: x.quantile(0.95)
        ).reset_index().set_index('kpi_name')
        rows_at_ts = rows_at_ts.merge(os020_global, left_on='kpi_name', right_index=True, how='left')
        # Determine anomaly type and severity
        def classify_and_severity(row):
            v = row['value']
            g95 = row['global_P95']
            g5  = row['global_P5']
            is_high = (v >= g95) if pd.notna(g95) else False
            is_low  = (v <= g5)  if pd.notna(g5) else False
            if is_high and is_low:
                atype = 'both'
            elif is_high:
                atype = 'high'
            elif is_low:
                atype = 'low'
            else:
                atype = 'none'
            # severity: absolute breach normalized as requested
            high_sev = safe_div(v - g95, v) if (v != 0 and pd.notna(g95)) else 0.0
            low_sev  = safe_div(g5 - v, v)  if (v != 0 and pd.notna(g5)) else 0.0
            if atype == 'high':
                sev = max(0.0, high_sev)
            elif atype == 'low':
                sev = max(0.0, low_sev)
            elif atype == 'both':
                sev = max(0.0, high_sev, low_sev)
            else:
                sev = 0.0
            return pd.Series([atype, float(sev), float(v), float(g95) if pd.notna(g95) else np.nan, float(g5) if pd.notna(g5) else np.nan])
        rows_at_ts[['anomaly_type','severity','value_at_timestamp','global_P95','global_P5']] = rows_at_ts.apply(classify_and_severity, axis=1)
        anomalies_at_earliest_df = rows_at_ts[['kpi_name','value_at_timestamp','global_P95','global_P5','anomaly_type','severity']].copy()
        # Filter out non-anomalous (atype == 'none')
        anomalies_at_earliest_df = anomalies_at_earliest_df[anomalies_at_earliest_df['anomaly_type'] != 'none']
        anomalies_at_earliest_df = anomalies_at_earliest_df.sort_values(by='severity', ascending=False).head(20).reset_index(drop=True)

# Final compact outputs: segments_df (all found segments), earliest_disk_anomaly_ts, anomalies_at_earliest_df (top 20)
segments_df, earliest_disk_anomaly_ts, anomalies_at_earliest_df

```
Out[7]:
```
Summary of disk KPI anomaly segments and cross-KPI anomalies at the earliest disk anomaly time (os_020):

1) Detected consecutive anomaly segments (value >= global_P95) for Disk_io_util and Disk_svctm during 2020-05-27 00:00:00+08 -> 00:30:00+08

kpi_name,segment_start_timestamp,segment_end_timestamp,length_in_minutes,max_value_in_segment,min_value_in_segment,mean_value_in_segment,global_P95,max_severity
Disk_io_util,2020-05-27 00:00:00+08:00,2020-05-27 00:30:00+08:00,31,0.333333,0.000000,0.010753,0.0000,1.000000
Disk_svctm,2020-05-27 00:00:00+08:00,2020-05-27 00:00:00+08:00,1,3.000000,3.000000,3.000000,2.2125,0.262500
Disk_svctm,2020-05-27 00:04:00+08:00,2020-05-27 00:04:00+08:00,1,29.780000,29.780000,29.780000,2.2125,0.925705

2) Earliest disk anomaly segment start across the two KPIs:
earliest_disk_anomaly_ts = 2020-05-27 00:00:00+08:00

3) KPIs anomalous at that exact timestamp (2020-05-27 00:00:00+08:00).
Columns: kpi_name,value_at_timestamp,global_P95,global_P5,anomaly_type,severity
Disk_wr_kbs,0.024133,0.386339,0.033900,low,0.404726
Sent_packets,17.333333,65.416667,24.333333,low,0.403846
Received_packets,15.333333,57.000000,21.500000,low,0.402174
Disk_svctm,3.000000,2.212500,0.000000,high,0.262500
Disk_rd_ios,0.000000,0.000000,0.000000,both,0.000000
Disk_io_util,0.000000,0.000000,0.000000,both,0.000000
Agent_ping,1.000000,1.000000,1.000000,both,0.000000
Disk_await,0.000000,2.090000,0.000000,low,0.000000
FS_max_util,83.000000,83.000000,83.000000,both,0.000000
Disk_rd_kbs,0.000000,0.000000,0.000000,both,0.000000
FS_total_space,299.962000,299.962000,299.962000,both,0.000000
ICMP_ping,1.000000,1.000000,1.000000,both,0.000000
Received_errors_packets,0.000000,0.000000,0.000000,both,0.000000
Page_pi,0.000000,0.000000,0.000000,both,0.000000
Sent_errors_packets,0.000000,0.000000,0.000000,both,0.000000
Shared_memory,358888.000000,358888.000000,358888.000000,both,0.000000

Notes:
- Disk_io_util had a continuous 31-minute anomaly segment (every minute in the window) vs. Disk_svctm had short spikes (1-minute segments; a large spike at 00:04 with high max_severity ≈0.9257).
- The earliest disk anomaly (00:00+08) coincides with many KPIs flagged low at that minute (e.g., Disk_wr_kbs, Sent_packets, Received_packets) and a small set flagged high (Disk_svctm). Severity shown is the normalized breach as requested.

The original code execution output of IPython Kernel is also provided below for reference:

(       kpi_name   segment_start_timestamp     segment_end_timestamp  length_in_minutes  max_value_in_segment  min_value_in_segment  mean_value_in_segment  global_P95  max_severity
0  Disk_io_util 2020-05-27 00:00:00+08:00 2020-05-27 00:30:00+08:00                 31              0.333333                  0.00               0.010753      0.0000      1.000000
1    Disk_svctm 2020-05-27 00:00:00+08:00 2020-05-27 00:00:00+08:00                  1              3.000000                  3.00               3.000000      2.2125      0.262500
2    Disk_svctm 2020-05-27 00:04:00+08:00 2020-05-27 00:04:00+08:00                  1             29.780000                 29.78              29.780000      2.2125      0.925705, Timestamp('2020-05-27 00:00:00+0800', tz='Asia/Shanghai'),                    kpi_name  value_at_timestamp     global_P95      global_P5 anomaly_type  severity
0               Disk_wr_kbs            0.024133       0.386339       0.033900          low  0.404726
1              Sent_packets           17.333333      65.416667      24.333333          low  0.403846
2          Received_packets           15.333333      57.000000      21.500000          low  0.402174
3                Disk_svctm            3.000000       2.212500       0.000000         high  0.262500
4               Disk_rd_ios            0.000000       0.000000       0.000000         both  0.000000
5              Disk_io_util            0.000000       0.000000       0.000000         both  0.000000
6                Agent_ping            1.000000       1.000000       1.000000         both  0.000000
7                Disk_await            0.000000       2.090000       0.000000          low  0.000000
8               FS_max_util           83.000000      83.000000      83.000000         both  0.000000
9               Disk_rd_kbs            0.000000       0.000000       0.000000         both  0.000000
10           FS_total_space          299.962000     299.962000     299.962000         both  0.000000
11                ICMP_ping            1.000000       1.000000       1.000000         both  0.000000
12  Received_errors_packets            0.000000       0.000000       0.000000         both  0.000000
13                  Page_pi            0.000000       0.000000       0.000000         both  0.000000
14      Sent_errors_packets            0.000000       0.000000       0.000000         both  0.000000
15            Shared_memory       358888.000000  358888.000000  358888.000000         both  0.000000)```
```